<a href="https://colab.research.google.com/github/Lucazere00/deep_learning/blob/main/weapons_detection_DETR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DETR**

In [ ]:
# Stampa il percorso del file pycocotools per verificarne l'installazione
!python -c "import pycocotools; print(pycocotools.__file__)"

/usr/local/lib/python3.12/dist-packages/pycocotools/__init__.py


In [ ]:
# Sostituisce 'np.float' con 'float' nel file cocoeval.py di pycocotools per compatibilità con NumPy più recenti
!sed -i 's/np.float/float/g' /usr/local/lib/python3.12/dist-packages/pycocotools/cocoeval.py

# **Dataset**

Il dataset è stato archiviato come Release su GitHub. Procediamo con il download del repository e la successiva estrazione (unzip) dei file. Questa procedura automatizzata permette di configurare l'ambiente di training in pochi secondi, evitando caricamenti manuali pesanti.

In [ ]:
# Clona il repository DETR da GitHub
!git clone https://github.com/facebookresearch/detr.git
# Scarica il dataset da GitHub
!wget https://github.com/Lucazere00/deep_learning/releases/download/dataset/dataset.zip
# Scompatta il dataset
!unzip -q dataset.zip -d /content/dataset
# Scarica il file del modello pre-addestrato DETR dalla release di GitHub
!wget https://github.com/Lucazere00/deep_learning/releases/download/model/detr_model_final.pth

Cloning into 'detr'...
remote: Enumerating objects: 265, done.
remote: Total 265 (delta 0), reused 0 (delta 0), pack-reused 265 (from 1)
Receiving objects: 100% (265/265), 21.19 MiB | 34.66 MiB/s, done.
Resolving deltas: 100% (120/120), done.
--2026-01-15 10:05:46--  https://github.com/Lucazere00/deep_learning/releases/download/dataset/dataset.zip
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1117631808/dba4f1b1-853b-483f-8e2d-b159e879db91?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-15T10%3A47%3A14Z&rscd=attachment%3B+filename%3Ddataset.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-15T09%3A46%3A51Z&ske=2026-01-15T10%3A47%3A14Z&sks=b&skv=2018-11-09&sig=b2w9LWoszyrHEYaV3XIkeC6Zu6ONr0Hkl9qQzpCO

In [ ]:
# Installa le librerie Cython e SciPy, spesso necessarie per pycocotools e altre dipendenze
!pip install cython scipy
#!pip install -U 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI' # Riga commentata per installazione di cocoapi, se necessaria

# **DETR**

In questa sezione viene configurato DETR (DEtection TRansformer) con backbone ResNet-50. Dopo aver clonato il repository ufficiale, il processo di addestramento è stato configurato per 80 epoche, con un decremento del learning rate (lr_drop) alla sessantesima epoca per stabilizzare la convergenza. Attualmente, il blocco di training è commentato poiché i pesi ottimizzati sono stati salvati e persistiti su Google Drive; questo permette di caricare direttamente il modello pronto, garantendo efficienza e risparmio di risorse computazionali.

In [ ]:
# Scarica i pesi pre-addestrati di DETR (ResNet-50) e li salva nella directory 'pretrained-weights'
!mkdir -p pretrained-weights
!wget https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth -O pretrained-weights/detr-r50-e632da11.pth

--2026-01-15 10:06:02--  https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.189.14, 3.163.189.96, 3.163.189.51, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.189.14|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 166618694 (159M) [application/octet-stream]
Saving to: ‘pretrained-weights/detr-r50-e632da11.pth’

pretrained-weights/ 100%[===================>] 158.90M  65.8MB/s    in 2.4s    

2026-01-15 10:06:05 (65.8 MB/s) - ‘pretrained-weights/detr-r50-e632da11.pth’ saved [166618694/166618694]



In [ ]:
# Codice per l'addestramento del modello DETR, attualmente commentato, perchè ho salvato i pesi dell'addestramento su drive
# !python /content/detr/main.py \
# --dataset_file coco \
# --coco_path /content/dataset/final_dataset \
# --output_dir /content/drive/MyDrive/DETR_outputs \
# --batch_size 2 \
# --epochs 80 \
# --lr_drop 60 \
# --lr_backbone 1e-5 \
# --resume /content/detr_model_final.pth

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'
import os
import torch
import json
from tqdm import tqdm
from torch import nn
from torchvision.models import resnet50
import torchvision.transforms as T
torch.set_grad_enabled(False);
import cv2
import time
from PIL import Image

import matplotlib.pyplot as plt

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # Imposta il dispositivo su GPU ('cuda') se disponibile, altrimenti su CPU
CONF_THRESHOLD = 0.5
# Definisco i path di input
IMAGE_DIR = "/content/dataset/dataset/test"
ANNOTATION_JSON = "/content/dataset/dataset/annotations/instances_test2017.json"
# Definisco i path di output
OUTPUT_DIR = "/content/dataset/dataset/predictions_detr"
ANNOTATED_DIR = os.path.join(OUTPUT_DIR, "annotated_images")
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "detr_test_predictions.json")

os.makedirs(ANNOTATED_DIR, exist_ok=True)

In [ ]:
# Definisce le classi personalizzate per il rilevamento di oggetti (persona e arma)
CLASSES = [
    'N/A',      # Classe 0 (Background)
    'person',   # Classe 1: Persona
    'weapon'    # Classe 2: Arma
]
# Mappa le etichette del modello alle category_id del formato COCO (utile se le id non coincidono)
LABEL_TO_COCO = {
    1: 1,  # La classe 'person' del modello corrisponde alla category_id 1 in COCO
    2: 2   # La classe 'weapon' del modello corrisponde alla category_id 2 in COCO
}

# Colori
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

In [ ]:
# Definizione delle trasformazioni standard per le immagini di input di PyTorch (normalizzazione media-deviazione standard)
transform = T.Compose([
    T.Resize(600),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [ ]:
# Carica il modello personalizzato addestrato
import sys
sys.path.append('./detr')

from models.detr import build


class Args:
    # Parametri del modello
    backbone = 'resnet50'
    dilation = False
    position_embedding = 'sine'
    num_classes = 2


    hidden_dim = 256
    nheads = 8
    num_encoder_layers = 6
    num_decoder_layers = 6
    dim_feedforward = 2048
    dropout = 0.1
    enc_layers = 6
    dec_layers = 6
    pre_norm = False
    num_queries = 100
    lr_backbone = 1e-5
    set_cost_class = 1
    set_cost_bbox = 5
    set_cost_giou = 2
    mask_loss_coef = 1
    dice_loss_coef = 1
    bbox_loss_coef = 5
    giou_loss_coef = 2
    eos_coef = 0.1
    # Altri parametri richiesti
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dataset_file = 'coco'
    masks = False
    aux_loss = True
    frozen_weights = None

args = Args()

model, criterion, postprocessors = build(args)

# Carica il checkpoint del modello addestrato
checkpoint_path = '/content/detr_model_final.pth'
# Carica il checkpoint, mappandolo sul dispositivo specificato
checkpoint = torch.load(checkpoint_path, map_location=args.device, weights_only=False)
# Carica lo stato del modello dal checkpoint.
model.load_state_dict(checkpoint, strict=False)
model.to(args.device)
model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


DETR(
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerDecoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, ou

In [ ]:
import json

# Apre e carica il file JSON delle annotazioni COCO di test
with open(ANNOTATION_JSON, "r") as f:
    coco_data = json.load(f)

# Crea un dizionario che mappa i nomi dei file immagine ai loro ID corrispondenti nel dataset COCO
image_name_to_id = {
    img["file_name"]: img["id"]
    for img in coco_data["images"]
}

In [ ]:
import cv2
import numpy as np

def draw_boxes(image, boxes, labels, scores): # Definisce una funzione per disegnare i bounding box su un'immagine
    img = image.copy()
    for box, label, score in zip(boxes, labels, scores):
        if score < CONF_THRESHOLD:
            continue
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
        text = f"{CLASSES[label]} {score:.2f}"
        cv2.putText(img, text, (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    return img

In [ ]:
import os
import shutil
# Reset dei Path di output per pulizia
OUTPUT_DIR = "/content/dataset/dataset/predictions_detr"
ANNOTATED_DIR = os.path.join(OUTPUT_DIR, "annotated_images")
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "detr_test_predictions.json")

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ANNOTATED_DIR, exist_ok=True)
print("Cartelle di output resettate e pronte.")

Cartelle di output resettate e pronte.


# **Inferenza**

Questa sezione gestisce l'inferenza del modello sulle immagini di test. Per ogni immagine, il modello identifica le classi target, filtra i risultati e salva le predizioni in un file JSON compatibile con il formato COCO. Questa procedura di standardizzazione è fondamentale per calcolare metriche come la Mean Average Precision (mAP).

In [ ]:
import os
import time
import torch
from PIL import Image
from tqdm import tqdm
import cv2

predictions = []
times = []

image_files = sorted(os.listdir(IMAGE_DIR))

# Inferenza
for img_name in tqdm(image_files, desc="Inferenza DETR"):

    if img_name not in image_name_to_id:
        continue

    img_path = os.path.join(IMAGE_DIR, img_name)
    image_id = image_name_to_id[img_name]

    pil_img = Image.open(img_path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)

    start = time.time()
    with torch.no_grad():
        outputs = model(img_tensor)
    times.append(time.time() - start)

    orig_size = torch.tensor([[pil_img.size[1], pil_img.size[0]]]).to(DEVICE)

    results = postprocessors["bbox"](outputs, orig_size)[0]

    scores = results["scores"]
    labels = results["labels"]
    boxes = results["boxes"]

    # --- FILTRO N/A e BACKGROUND ---
    keep = (labels > 0) & (labels < len(CLASSES)) & (scores >= 0.5)
    scores = scores[keep]
    labels = labels[keep]
    boxes = boxes[keep]

    # JSON COCO
    # Prepara le predizioni nel formato COCO JSON
    for score, label, box in zip(scores, labels, boxes):
        x1, y1, x2, y2 = box.tolist()
        predictions.append({
            "image_id": image_id,
            "category_id": LABEL_TO_COCO[label.item()],
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "score": score.item()
        })

    # --- IMMAGINE ANNOTATA ---

    img_cv = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    annotated = draw_boxes(
        img_cv,
        boxes.cpu().numpy(),
        labels.cpu().numpy(),
        scores.cpu().numpy()
    )

    save_path = os.path.join(ANNOTATED_DIR, img_name)
    cv2.imwrite(save_path, cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

Inferenza DETR: 100%|██████████| 500/500 [00:24<00:00, 20.52it/s]


In [ ]:
import json

# Salvo le predizioni COCO
with open(OUTPUT_JSON, "w") as f:
    json.dump(predictions, f, indent=2)

print("\n--- COMPLETATO ---")
print(f"Predizioni totali: {len(predictions)}")
print(f"Tempo medio per immagine: {sum(times)/len(times):.3f}s")
print(f"JSON salvato in: {OUTPUT_JSON}")
print(f"Immagini annotate in: {ANNOTATED_DIR}")


--- COMPLETATO ---
Predizioni totali: 1621
Tempo medio per immagine: 0.026s
JSON salvato in: /content/dataset/dataset/predictions_detr/detr_test_predictions.json
Immagini annotate in: /content/dataset/dataset/predictions_detr/annotated_images


# **Risultati**

I risultati ottenuti con DETR (ResNet-50) mostrano un mAP di 0.600. Sebbene il modello dimostri una buona capacità di generalizzazione sugli oggetti di grandi dimensioni (AP Large 0.723), si riscontra un limite critico nel rilevamento di piccoli oggetti, con un AP Small prossimo allo zero (0.002).

Questo fenomeno è riconducibile all'architettura nativa di DETR, che non utilizza una gerarchia di feature map  e tende a perdere i dettagli spaziali degli oggetti più piccoli durante il passaggio nel Transformer Encoder. Tuttavia, l'AP @0.50 di 0.744 conferma che il modello identifica correttamente la presenza delle classi target nella maggior parte dei casi, pur faticando nella precisione millimetrica della localizzazione rispetto ai modelli precedentemente analizzati.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


cocoGt = COCO(ANNOTATION_JSON)

cocoDt = cocoGt.loadRes(OUTPUT_JSON)


cocoEval = COCOeval(cocoGt, cocoDt, "bbox")
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.46s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.600
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.744
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.635
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.002
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.094
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.723
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.569
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.647
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets